# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


# Bayesian Estimation of a User Ability Parameter from Item Responses


---

## 1. Visualizing the Mechanics of the 2PL IRT Model

The probability of a correct response under the 2PL model is defined as:

$$P(Y_i=1 \mid \Theta=\theta) = p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

To visualize this model, we can plot the Item Response Function (IRF) for two configurations:
1. **Varying difficulties ($b_i \in \{-1, 0, 1\}$)** with a fixed high discrimination ($a_i = 1.5$)
2. **Low discrimination ($a_i = 0.5$)** with a fixed average difficulty ($b_i = 0$)



In [3]:
import numpy as np
import plotly.graph_objects as go

# Define theta grid
theta = np.linspace(-4, 4, 200)

def p_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Create Plotly curves
fig = go.Figure()

# 1. Fixed high discrimination, varying difficulty
fig.add_trace(go.Scatter(x=theta, y=p_correct(theta, 1.5, -1), name="a = 1.5, b = -1 (Easy)", line=dict(color='green', width=2)))
fig.add_trace(go.Scatter(x=theta, y=p_correct(theta, 1.5, 0), name="a = 1.5, b = 0 (Moderate)", line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=theta, y=p_correct(theta, 1.5, 1), name="a = 1.5, b = 1 (Hard)", line=dict(color='red', width=2)))

# 2. Low discrimination, moderate difficulty
fig.add_trace(go.Scatter(x=theta, y=p_correct(theta, 0.5, 0), name="a = 0.5, b = 0 (Low Discrim)", line=dict(color='purple', dash='dash', width=2)))

fig.update_layout(
    title="Item Response Functions (IRF) for Various 2PL Parameters",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y = 1)",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.show()

### Interpretation of Horizontal Shifting
The difficulty parameter $b_i$ acts strictly as a **location parameter** (or horizontal shift) along the ability axis:
* At exactly $\theta = b_i$, the probability of a correct answer is always exactly $0.5$, since:
  $$p_i(b_i) = \frac{1}{1 + e^{-a_i(b_i - b_i)}} = \frac{1}{1 + e^0} = 0.5$$
* Increasing $b_i$ shifts the entire Sigmoid curve to the **right**. This requires the student to possess a higher latent ability to achieve the same probability of success, denoting a harder item.
* Decreasing $b_i$ shifts the curve to the **left**, adapting the instrument to lower ability ranges (signifying an easier item).

---

## 2. Sequential Likelihood Contribution

For a single item response $y_k \in \{0,1\}$, the likelihood contribution at step $k$ given the latent ability $\theta$ is modeled as a Bernoulli trial:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Substituting the explicit 2PL formula yields:

$$L(y_k \mid \theta) = \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k} = \frac{e^{-a_k(\theta-b_k)(1-y_k)}}{1 + e^{-a_k(\theta - b_k)}}$$

### Joint Likelihood Function
Assuming **local independence** (meaning responses are conditionally independent given the user's underlying ability parameter $\theta$), the joint likelihood for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual steps:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

---

## 3. Mathematical Formulation of the Running Update

By applying Bayes' Theorem sequentially, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting our single-item 2PL likelihood expression provides the running recursive relationship up to a normalization constant:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \frac{e^{-a_k(\theta-b_k)(1-y_k)}}{1 + e^{-a_k(\theta - b_k)}} \right] f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

---

## 4. Dynamic Shifting Behavior

When a user answers a highly difficult item correctly ($y_k = 1$, large $b_k$):

1. **Likelihood Shape:** The current likelihood reduces to:
   $$L(1 \mid \theta) = p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$$
   Because $b_k$ is large, the midpoint ($0.5$ probability threshold) is located far down the positive end of the axis.
2. **Gradient Effect:** This function is monotonically increasing. It penalizes low $\theta$ ranges near $0$, and transitions steeply to $1$ only in the high ability ranges.
3. **Posterior Shift:** Multiplying the existing prior distribution $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ by this right-heavy, upward-sloping step penalizes low ability assumptions and heavily weights the upper region. Consequently, the peak (mode) of the running posterior density shifts **significantly to the right**, rewarding the system with an aggressive upward revision of the student's ability estimate.

---

## 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the scale of the slope at the difficulty threshold $b_k$, directly dictating how much information is extracted:

* **High Discrimination (Large $a_k$):** The likelihood curve becomes very steep, acting almost like a hard threshold classifier. It sharply partitions the parameter space into "highly likely" vs. "highly unlikely". Multiplying by this distinct slope rapidly narrows the posterior distribution, decreasing variance and sharply increasing the certainty (sharpness) of the measurement.
* **Low Discrimination (Small $a_k \approx 0$):** The curve becomes nearly flat (approaching a uniform probability of $0.5$ regardless of skill). Multiplying the existing distribution by a flat line preserves its original shape and width, imparting virtually zero new information and leaving certainty unchanged.

---

## 6. Numerical Implementation of a Running Grid

Because the denominator (the marginal likelihood) lacks a closed-form analytical solution, we compute the continuous density on a fixed discrete grid.

### Algorithmic Sequence
1. **Grid Setup:** Define a discrete array of $M$ equally spaced points covering a realistic standard boundary:
   $$\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$$
   running from $-4$ to $+4$, with spacing $\Delta \theta = \theta_j - \theta_{j-1}$.
2. **Prior State:** Initialize the vector using the standard normal distribution formula evaluated at each point:
   $$P_0(\theta_j) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_j^2}{2}\right)$$
   Normalize it over the grid such that:
   $$\sum_{j=1}^M P_0(\theta_j)\Delta\theta = 1$$
3. **Dynamic Loop Update:** For each new question observation $(y_k, a_k, b_k)$:
   * Compute the unnormalized likelihood value across all grid indices:
     $$\tilde{L}_k(\theta_j) = [p_k(\theta_j)]^{y_k} [1 - p_k(\theta_j)]^{1 - y_k}$$
   * Multiply element-wise to form the raw new posterior:
     $$\tilde{P}_k(\theta_j) = \tilde{L}_k(\theta_j) \cdot P_{k-1}(\theta_j)$$
   * **Computational Normalization:** Implement a numerical approximation of the integral using a Riemann sum to enforce proper density scaling:
     $$C = \sum_{j=1}^M \tilde{P}_k(\theta_j) \Delta \theta \quad \implies \quad P_k(\theta_j) = \frac{\tilde{P}_k(\theta_j)}{C}$$

---

## 7. Evaluating Convergence over the Timeline

The Python code block below sets up a sequence of $n = 20$ dynamic interactions for a user whose true hidden latent ability is locked at $\theta_{\text{true}} = 0.75$. Run this script in a code cell to observe the running updates.

In [4]:
import numpy as np
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(42)

# Simulation Parameters
n_items = 20
theta_true = 0.75

# 1. Generate random items: b ~ N(0, 1), a ~ Uniform(0.5, 2.0)
b_params = np.random.normal(0, 1, n_items)
a_params = np.random.uniform(0.5, 2.0, n_items)

# 2. Simulate progressive user responses
def p_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

probabilities = p_correct(theta_true, a_params, b_params)
responses = (np.random.uniform(0, 1, n_items) < probabilities).astype(int)

# 3. Running Grid Bayesian Update Setup
grid_points = 1000
theta_grid = np.linspace(-4, 4, grid_points)
delta_theta = theta_grid[1] - theta_grid[0]

# Prior: Standard Normal N(0, 1)
prior = (1 / np.sqrt(2 * np.pi)) * np.exp(-theta_grid**2 / 2)
prior /= np.sum(prior) * delta_theta  # Initial normalization

# Trace holders for step 0 up to n
est_mean = [0.0]  # EAP starts at prior mean
est_map = [0.0]   # MAP starts at prior mode

current_posterior = prior.copy()

for k in range(n_items):
    # Fetch parameters for current item step
    a, b, y = a_params[k], b_params[k], responses[k]

    # Calculate single item likelihood over the entire grid
    p_grid = p_correct(theta_grid, a, b)
    likelihood = (p_grid**y) * ((1 - p_grid)**(1 - y))

    # Perform Bayesian Update and apply numerical normalization
    current_posterior = likelihood * current_posterior
    current_posterior /= np.sum(current_posterior) * delta_theta

    # Calculate Expected A Posteriori (EAP) / Mean
    post_mean = np.sum(theta_grid * current_posterior) * delta_theta

    # Calculate Maximum A Posteriori (MAP) / Mode
    post_map = theta_grid[np.argmax(current_posterior)]

    est_mean.append(post_mean)
    est_map.append(post_map)

# 4. Visualization
steps = np.arange(0, n_items + 1)
fig = go.Figure()

# True ability benchmark line
fig.add_trace(go.Scatter(x=[0, n_items], y=[theta_true, theta_true],
                         name="True Ability (θ = 0.75)",
                         line=dict(color='black', dash='dash')))

# Dynamic tracking curves
fig.add_trace(go.Scatter(x=steps, y=est_mean, name="Posterior Mean (EAP)",
                         line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=steps, y=est_map, name="Maximum A Posteriori (MAP)",
                         line=dict(color='orange', width=2, dash='dot')))

fig.update_layout(
    title="Bayesian Ability Parameter Estimation Convergence over 20 Items",
    xaxis_title="Item Step (k)",
    yaxis_title="Estimated Ability (θ)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    template="plotly_white",
    legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99)
)
fig.show()

### Convergence & Confidence Analysis
* **Tracking Convergence:** In the early sequence items ($k < 6$), both the Posterior Mean and the MAP estimates display pronounced volatility. Because the data pool is small, individual correct or incorrect answers cause abrupt shifts in the distribution. As the sequence extends toward $k = 20$, the variance of the error drops, and the lines steadily converge around the true benchmark value of $0.75$.
* **Information Accumulation:** Every added question provides an independent signal that compounds into the joint likelihood function. As the data space expands, the likelihood term naturally begins to dominate the initial normal prior distribution, effectively centering the density mass exactly over the student's true underlying competence level.
* **Platform Certainty:** As the number of items increases, the variance (spread) of the hidden posterior density narrows significantly. This mathematical narrowing ensures that successive updates produce progressively minor corrections. The stable flattening of the line towards the target demonstrates that the platform's measurement confidence is high, signaling it has accumulated sufficient data to safely lock onto the user's latent capability.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

# Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates


---

## 1. Structural Probability and Properties of the Beta Distribution

The Beta distribution acts as a flexible conjugate prior over continuous parameters bounded between $[0, 1]$. Its Probability Density Function (PDF) is defined as:

$$f(\theta \mid \alpha, \beta) = \frac{1}{\mathrm{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$

where the Beta function is defined using the Gamma function $\Gamma(x)$:

$$\mathrm{B}(\alpha, \beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha + \beta)}$$

To understand how the shape parameters $\alpha$ and $\beta$ govern the density, we can plot three key states:
*   **Uninformative State:** $\alpha = 1, \beta = 1$
*   **Right-Skewed State:** $\alpha = 2, \beta = 8$
*   **Left-Skewed State:** $\alpha = 8, \beta = 2$



In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Define theta grid
theta = np.linspace(0, 1, 500)

# Create Plotly curves
fig = go.Figure()

# Plot the three states
fig.add_trace(go.Scatter(x=theta, y=stats.beta.pdf(theta, 1, 1), name="α = 1, β = 1 (Uniform / Uninformative)", line=dict(color='gray', dash='dash', width=2)))
fig.add_trace(go.Scatter(x=theta, y=stats.beta.pdf(theta, 2, 8), name="α = 2, β = 8 (Right-skewed / Low CTR)", line=dict(color='red', width=2.5)))
fig.add_trace(go.Scatter(x=theta, y=stats.beta.pdf(theta, 8, 2), name="α = 8, β = 2 (Left-skewed / High CTR)", line=dict(color='green', width=2.5)))

fig.update_layout(
    title="Probability Density Function (PDF) of different Beta Distributions",
    xaxis_title="Latent Click Probability (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)
fig.show()

### Interpretation of Center of Mass Shifts
The parameters $\alpha$ and $\beta$ can be intuitively viewed as **prior pseudo-counts** of successes (clicks) and failures (non-clicks), respectively:
*   When $\alpha = \beta = 1$, the density is completely flat (a Uniform distribution). The platform has no preference for any value in the range $[0, 1]$.
*   When $\beta > \alpha$ (such as $\alpha=2, \beta=8$), the center of mass shifts to the **left (right-skewed)**. This represents a strong prior belief that the advertisement has a low CTR (peaking near $\frac{\alpha-1}{\alpha+\beta-2} = 0.125$).
*   When $\alpha > \beta$ (such as $\alpha=8, \beta=2$), the center of mass shifts to the **right (left-skewed)**. This signifies a strong prior belief that the advertisement is highly engaging and has a high CTR.

---

## 2. Sequential Likelihood and Joint History

For a single user interaction observation $y_k \in \{0, 1\}$ at step $k$, the outcome is modeled as a Bernoulli trial conditional on the underlying parameter $\theta$:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Assuming **conditional independence** of user actions given $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is simply the product of individual Bernoulli likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

Letting $s_k = \sum_{i=1}^k y_i$ represent the total number of clicks (successes) and $f_k = k - s_k$ represent the total number of non-clicks (failures) up to step $k$, the joint likelihood is elegantly summarized as:

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{s_k} (1 - \theta)^{f_k}$$

---

## 3. Closed-Form Analytical Updates (Beta-Binomial Conjugacy Proof)

We wish to derive the recursive update relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$.

By Bayes' Theorem, the posterior is proportional to the product of the likelihood of the new single observation $y_k$ and the prior (which is the previous step's posterior $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Assume that at step $k-1$, the distribution is a Beta distribution:

$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$

We now multiply this prior by the likelihood of our new observation $y_k$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Combining bases with common exponents:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This structural functional form is immediately recognizable as the kernel of a **Beta distribution** with updated parameters. Because the posterior distribution stays in the same probability family as the prior, the Beta distribution is a **conjugate prior** for the Bernoulli/Binomial likelihood.

### Closed-Form Parameter Updates
The exact recursive equations for the updated parameters are:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

### Posterior Mean Calculation
The expected value of a random variable distributed as $\text{Beta}(\alpha_k, \beta_k)$ yields the Posterior Mean:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_{k-1} + y_k}{\alpha_{k-1} + \beta_{k-1} + 1}$$

---

## 4. Dynamic Shifting Mechanics

Sequential Bayesian updates shift the distribution's mass depending on user interactions:
*   **An Observed Click ($y_k = 1$):** Mathematically, we add $1$ to the success parameters ($\alpha_k = \alpha_{k-1} + 1$, while $\beta_k = \beta_{k-1}$). This shifts the peak of the probability distribution to the **right** (toward $1$).
*   **An Observed Non-Click ($y_k = 0$):** We add $1$ to the failure parameters ($\alpha_k = \alpha_{k-1}$, while $\beta_k = \beta_{k-1} + 1$). This shifts the peak of the distribution to the **left** (toward $0$).

### Conjugate vs. Non-Conjugate Updates
| Attribute | Conjugate Setup (Beta-Binomial) | Non-Conjugate Setup (e.g., 2PL IRF Model) |
| :--- | :--- | :--- |
| **Posterior Computation** | **Algebraic:** Shape parameters updated via simple addition ($\alpha_k = \alpha_{k-1} + y_k$). | **Numerical:** Requires grid evaluation or approximation methods (MCMC, Laplace). |
| **Normalizing Constant** | **Analytical:** Normalization uses the closed-form Beta function $\mathrm{B}(\alpha_k, \beta_k)$. | **Numerical Integration:** Requires approximating a complex integral $\int P(\mathbf{Y} \mid \theta) f(\theta) d\theta$. |
| **Speed/Scale** | Highly scalable; ideal for instantaneous real-time streaming updates. | Computationally intensive; scales poorly on extremely fast, high-volume data streams. |

---

## 5. Running Point Estimators

From our updated analytical parameters $\alpha_k$ and $\beta_k$, we evaluate the running estimates at any step $k$ without needing to run integration loops:

### Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
Also known as the Expected A Posteriori (EAP) estimate. It represents the center of mass of the updated distribution:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

### Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)
Also known as the mode. It represents the single most likely value (the highest peak) of the posterior density. For $\alpha_k > 1$ and $\beta_k > 1$:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

---

## 6. Performance Tracking and Convergence Analysis

The code block below runs a simulation of $n = 100$ impressions for an advertisement with an actual click-through rate of $\theta_{\text{true}} = 0.35$.



In [6]:
import numpy as np
import plotly.graph_objects as go

# Set random seed for scientific consistency
np.random.seed(42)

# Simulation Parameters
n_impressions = 100
theta_true = 0.35

# 1. Generate sequential user clicks (Bernoulli trials with p = theta_true)
uniform_draws = np.random.uniform(0, 1, n_impressions)
responses = (uniform_draws < theta_true).astype(int)

# 2. Sequential Beta-Binomial Updating
# Initialize flat prior: Beta(1, 1)
alpha_val = 1
beta_val = 1

# Traces starting at Step 0
est_mean = [alpha_val / (alpha_val + beta_val)]  # Prior mean = 0.5
# Prior MAP mode for Beta(1,1) is technically undefined/uniform, we initialize at 0.5
est_map = [0.5]

# Sequential updates
for k in range(n_impressions):
    y = responses[k]

    # Exact conjugate updates
    alpha_val += y
    beta_val += (1 - y)

    # Compute running estimators
    post_mean = alpha_val / (alpha_val + beta_val)

    # Map formula requires alpha, beta > 1 to avoid boundary peaks
    if alpha_val > 1 and beta_val > 1:
        post_map = (alpha_val - 1) / (alpha_val + beta_val - 2)
    else:
        post_map = post_mean # Fallback to mean in early trials if needed

    est_mean.append(post_mean)
    est_map.append(post_map)

# 3. Interactive Plotly Visualization
steps = np.arange(0, n_impressions + 1)
fig = go.Figure()

# True CTR Line
fig.add_trace(go.Scatter(x=[0, n_impressions], y=[theta_true, theta_true],
                         name="True CTR (θ = 0.35)",
                         line=dict(color='black', dash='dash', width=1.5)))

# Bayesian Estimators
fig.add_trace(go.Scatter(x=steps, y=est_mean, name="Posterior Mean (EAP)",
                         line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=steps, y=est_map, name="Maximum A Posteriori (MAP)",
                         line=dict(color='orange', width=2, dash='dot')))

fig.update_layout(
    title="Beta-Binomial Bayesian Updating Convergence over 100 Impressions",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated Click-Through Rate (θ)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=10),
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)
fig.show()

### Convergence & Prior Influence Analysis

*   **Convergence Path:** During the early steps ($k < 20$), both the Posterior Mean and the MAP estimators bounce around aggressively. This is because the overall sample size is low, making the estimators highly sensitive to short bursts of successive clicks or non-clicks.
*   **Diminishing Deviations:** As the impression count $k$ approaches $100$, the estimators stabilize and converge closely around the true parameter value of $0.35$. The deviation between the two estimators also drops to near zero because the updated distribution is narrowing into a sharp, symmetric Gaussian-like shape centered at the mean.
*   **Prior vs. Data Evidence:** At step $0$, the estimators are heavily dependent on our chosen prior ($\text{Beta}(1,1)$). However, as observations accumulate, the joint likelihood function $\theta^{s_k} (1-\theta)^{f_k}$ grows exponentially in magnitude compared to the static prior. By step $100$, the data has effectively "swamped the prior", demonstrating that sequential Bayesian updates are naturally self-correcting—the initial prior assumption becomes irrelevant as the sample size grows.

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

# Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates


---

## 1. Prior Belief Boundaries

Before data collection starts, engineers assume the component is highly likely to be pristine. We model this prior belief using a bounded Beta distribution:

$$\Theta \sim \text{Beta}(\alpha, \beta) \quad \text{where} \quad \alpha = 8, \;\; \beta = 1.5 \quad \text{over the domain} \quad \theta \in (0, 1]$$

### Analytical Prior Mean
The expected value of a standard Beta distribution is:

$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421$$

### Engineering Suitability
This distribution is highly appropriate as an initial prior because:
1. **Physical Bounds:** The Beta distribution naturally constrains the domain to $(0, 1]$, which perfectly matches the physical stiffness efficiency boundaries (a structure cannot be more than $100\%$ pristine, nor can its stiffness be negative).
2. **Optimistic Bias:** A higher concentration of density ($a = 8 > b = 1.5$) places the peak close to $1.0$, reflecting our engineering expectation that newly deployed or certified structures are highly likely to be healthy while maintaining a smooth, decreasing probability tail for pre-existing damage.


In [7]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Define grid matching the physical domain
theta_grid = np.linspace(0.01, 1.0, 1000)

# Prior density evaluation
prior_pdf = stats.beta.pdf(theta_grid, 8, 1.5)

# Plotting the initial prior
fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_pdf, name="Beta(8, 1.5) Prior", line=dict(color='blue', width=3.5)))
fig.update_layout(
    title="Initial Prior Belief: Bounded Beta(8, 1.5) Distribution",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    xaxis=dict(range=[0, 1.05]),
    yaxis=dict(range=[0, max(prior_pdf) * 1.1])
)
fig.show()

---

## 2. Structural Likelihood Formulation

The measurement model at step $k$ is given by:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

This is equivalent to:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

Thus, the natural logarithm of the measurement is normally distributed:

$$\ln(y_k) \sim \mathscr{N}\left(\ln(\theta \cdot K_{\text{nominal}}), \; \sigma^2\right)$$

Using the standard change of variables formula for probability densities, $f_{Y_k}(y_k) = f_{\ln(Y_k)}(\ln y_k) \left| \frac{d\ln y_k}{dy_k} \right| = f_{\ln(Y_k)}(\ln y_k) \frac{1}{y_k}$, we obtain the log-normal likelihood contribution for a single isolated measurement:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln y_k - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

### Joint Likelihood Function
Assuming that sensor noise is conditionally independent across inspection intervals (local independence), the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln y_i - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

---

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

### Absence of an Analytical Closed-Form Solution
A conjugate relationship does not exist here because the structural likelihood is **log-normal** (governed by exponentials of logarithmic terms containing the product of $\theta$), while the prior is a **Beta distribution** (governed by power-law polynomial terms $\theta^{\alpha-1}(1-\theta)^{\beta-1}$).

When multiplied, the term $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ does not algebraically combine with the exponent containing $\ln(\theta)$ in any way that matches a standard probability distribution. Consequently, the posterior distribution lacks an analytical closed-form representation, necessitating numerical grid integration.

### Recursive Relationship
By applying Bayes' Theorem, the recursive update relationship up to a proportionality constant is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting our likelihood, we obtain:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \frac{1}{\theta} \exp\left(-\frac{\left(\ln y_k - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2}\right) \right] \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

*(Note: The $1/y_k$ factor from the likelihood is dropped as it is a multiplicative constant with respect to $\theta$.)*

---

## 4. Running Point Estimates

Since the resulting posterior distribution is non-standard, we define our estimators via definite integrals over the physical domain $(0, 1]$:

### Running Posterior Mean (EAP)
The Expected A Posteriori (EAP) estimator represents the center of mass of the updated posterior distribution:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{y}^{(k)}] = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

### Running Maximum A Posteriori (MAP)
The MAP estimator represents the mode of the posterior distribution—the point where the probability density is maximized:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

## 5. Algorithmic Grid Approximation and Normalization

To perform these updates in real-time, we discretize the parameter domain onto a fine grid:

1. **Grid Discretization:** Choose a highly dense grid of $M$ equally spaced points (e.g., $M = 1000$) over the physical interval $[\theta_{\min}, \theta_{\max}] = [0.001, 1.0]$. The step size is $\Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}$. The lower limit is set slightly above zero ($\theta_{\min} = 0.001$) to avoid taking $\ln(0)$, preventing non-physical computations.
2. **Prior Initialization:** Compute the prior density vector $\mathbf{P}_0$ over the grid where $P_0(\theta_j) = \text{Beta.PDF}(\theta_j, 8, 1.5)$. Normalize it using the trapezoidal rule:
   $$C_0 = \sum_{j=1}^{M-1} \frac{P_0(\theta_j) + P_0(\theta_{j+1})}{2} \Delta\theta \quad \implies \quad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{C_0}$$
3. **Sequential Processing Loop:** Upon receiving a new measurement $y_k$:
   * Calculate the likelihood vector over each grid point:
     $$L_k(\theta_j) = \exp\left( -\frac{(\ln y_k - \ln(\theta_j \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$
   * Compute the raw unnormalized posterior vector:
     $$\tilde{P}_k(\theta_j) = L_k(\theta_j) \cdot P_{k-1}(\theta_j)$$
   * **Trapezoidal Normalization:** Compute the normalizing constant $C_k$ and normalize the vector:
     $$C_k = \frac{\Delta\theta}{2} \left[ \tilde{P}_k(\theta_1) + 2\sum_{j=2}^{M-1} \tilde{P}_k(\theta_j) + \tilde{P}_k(\theta_M) \right] \quad \implies \quad P_k(\theta_j) = \frac{\tilde{P}_k(\theta_j)}{C_k}$$
   * **Calculate Estimators:**
     $$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \sum_{j=1}^{M-1} \frac{\theta_j P_k(\theta_j) + \theta_{j+1} P_k(\theta_{j+1})}{2} \Delta\theta$$
     $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \theta_{j^*} \quad \text{where} \quad j^* = \arg\max_{j} P_k(\theta_j)$$

---

## 6. Performance Tracking and Degradation Convergence Analysis


In [8]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set seed for reproducible simulation
np.random.seed(42)

# Structural Parameters
n_steps = 15
theta_true = 0.68
K_nominal = 50.0  # kN/mm
sigma = 0.15

# 1. Simulate the Sensor Stream based on physics model
epsilon = np.random.normal(0, sigma, n_steps)
y_measurements = theta_true * K_nominal * np.exp(epsilon)

# 2. Grid Update Setup
M = 1000
theta_grid = np.linspace(0.001, 1.0, M)
delta_theta = theta_grid[1] - theta_grid[0]

# Prior: Beta(8, 1.5) normalized using trapezoidal integration
prior_pdf = stats.beta.pdf(theta_grid, 8, 1.5)
prior_pdf /= np.trapezoid(prior_pdf, dx=delta_theta)

# History dictionaries to track estimators
est_mean = [np.trapezoid(theta_grid * prior_pdf, dx=delta_theta)]  # Step 0 Prior Mean
est_map = [theta_grid[np.argmax(prior_pdf)]]                       # Step 0 Prior MAP

# Track full posteriors at specified milestones
milestones = {0: prior_pdf.copy()}
current_posterior = prior_pdf.copy()

# Sequential processing loop
for k in range(1, n_steps + 1):
    yk = y_measurements[k - 1]

    # Calculate log-normal likelihood over the grid (ignoring 1/yk factor)
    log_mean = np.log(theta_grid * K_nominal)
    likelihood = np.exp(-((np.log(yk) - log_mean) ** 2) / (2 * (sigma ** 2)))

    # Update and normalize posterior
    current_posterior = likelihood * current_posterior
    norm_factor = np.trapezoid(current_posterior, dx=delta_theta)
    current_posterior /= norm_factor

    # Calculate estimators
    post_mean = np.trapezoid(theta_grid * current_posterior, dx=delta_theta)
    post_map = theta_grid[np.argmax(current_posterior)]

    est_mean.append(post_mean)
    est_map.append(post_map)

    # Store milestone states
    if k in [1, 2, 5, 10, 15]:
        milestones[k] = current_posterior.copy()

# ==================== VISUALIZATION 1 ====================
fig1 = go.Figure()
colors = {0: 'gray', 1: 'pink', 2: 'purple', 5: 'orange', 10: 'blue', 15: 'green'}
for ms, density in milestones.items():
    dash_style = 'dash' if ms == 0 else 'solid'
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=density,
        name=f"Step k = {ms}",
        line=dict(color=colors[ms], width=2.5, dash=dash_style)
    ))

fig1.update_layout(
    title="Sequential Evolution of the Stiffness Posterior Distribution (Milestones)",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    xaxis=dict(range=[0.3, 1.05])
)
fig1.show()

# ==================== VISUALIZATION 2 ====================
steps = np.arange(0, n_steps + 1)
fig2 = go.Figure()

# True Target Line
fig2.add_trace(go.Scatter(
    x=[0, n_steps], y=[theta_true, theta_true],
    name="True Stiffness (θ = 0.68)",
    line=dict(color='black', dash='dash', width=2)
))

# Bayesian Trajectories
fig2.add_trace(go.Scatter(x=steps, y=est_mean, name="Posterior Mean (EAP)", line=dict(color='blue', width=2.5)))
fig2.add_trace(go.Scatter(x=steps, y=est_map, name="MAP Estimate", line=dict(color='orange', width=2.5, dash='dot')))

fig2.update_layout(
    title="Convergence Timeline of Stiffness Estimators over 15 Measurements",
    xaxis_title="Inspection Milestone Step (k)",
    yaxis_title="Estimated Stiffness Factor (θ)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(range=[0.4, 0.95]),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)
fig2.show()

### Convergence & Structural Safety Analysis

*   **Overcoming the Prior Bias:** The initial prior distribution ($\text{Beta}(8, 1.5)$) heavily favors the healthy state ($\theta \approx 0.84$). In the plots, we can see that it takes **only 2 to 3 sensor readings** ($k = 2$ or $3$) to pull the estimators completely away from this highly optimistic prior bias. The evidence introduced by the noisy log-normal measurements rapidly shifts the posterior's center of mass toward the damage state.
*   **Narrowing of Density Curves:** As $k$ progresses from $1$ to $15$, the posterior distribution transitions from a wide, spread-out shape to an extremely narrow, sharp peak centered tightly around $0.68$. This narrowing represents a continuous **drop in the posterior variance**, indicating that the platform's mathematical certainty is increasing with every observation.
*   **Engineering Implications for Safety Thresholds:**
    1. In Structural Health Monitoring, a wide posterior density represents high parameter uncertainty, forcing safety engineers to maintain broad, costly tolerance margins.
    2. By tracking the narrowing of the posterior distribution, we can construct precise **Bayesian Credible Intervals** (e.g., $95\%$ certainty intervals). Once the upper boundary of this credible interval drops below an established safety threshold (such as $0.70$ remaining stiffness), the system can confidently trigger automatic maintenance flags. This reduces the risk of costly false alarms while guaranteeing immediate action before structural fatigue reaches catastrophic limits.

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

In [9]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.graph_objects as go

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(
            n_components=self.n_components,
            random_state=self.random_state,
            covariance_type='full'
        )
        self.X_train_scaled = None
        self.X_val_scaled = None
        self.X_train_raw = None
        self.X_val_raw = None

    def load_and_preprocess_data(self):
        # Load the CC DATA dataset from a public raw github source
        url = "https://raw.githubusercontent.com/jplatos/Fundamentals-of-Machine-Learning/main/CCGENERAL.csv"
        df = pd.read_csv(url)

        # Select two continuous features of interest
        features = ['PURCHASES', 'CREDIT_LIMIT']
        df_subset = df[features].dropna() # Drop rows with missing values

        # Save raw matrices
        X_raw = df_subset.values

        # 80/20 train/validation split
        X_train, X_val = train_test_split(
            X_raw, test_size=0.20, random_state=self.random_state
        )

        # Standardize features
        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_val_scaled = self.scaler.transform(X_val)

        # Convert scaled values back to DataFrame for easier plotting
        self.X_train_raw = pd.DataFrame(X_train, columns=features)
        self.X_val_raw = pd.DataFrame(X_val, columns=features)

        print(f"Data successfully preprocessed.")
        print(f"Training set shape: {self.X_train_scaled.shape}, Validation set shape: {self.X_val_scaled.shape}")

    def fit(self):
        self.gmm.fit(self.X_train_scaled)
        print("--- EM Algorithm Execution ---")
        print(f"Converged: {self.gmm.converged_}")
        print(f"Number of iterations to converge: {self.gmm.n_iter_}")

    def evaluate(self):
        # Calculate average log-likelihood on validation set
        avg_log_likelihood = self.gmm.score(self.X_val_scaled)
        print(f"Validation Set Average Log-Likelihood: {avg_log_likelihood:.4f}")
        return avg_log_likelihood

    def plot_raw_density(self):
        # Empirical 2D Density Heatmap with marginals
        fig = px.density_heatmap(
            self.X_train_raw, x="PURCHASES", y="CREDIT_LIMIT",
            marginal_x="histogram", marginal_y="histogram",
            title="Empirical 2D Density Heatmap of Credit Card Behaviors",
            labels={'PURCHASES': 'Purchases ($)', 'CREDIT_LIMIT': 'Credit Limit ($)'},
            template="plotly_white"
        )
        return fig

    def _generate_contour_data(self):
        # Determine plot bounds
        x_min, x_max = self.X_train_raw['PURCHASES'].min(), self.X_train_raw['PURCHASES'].max()
        y_min, y_max = self.X_train_raw['CREDIT_LIMIT'].min(), self.X_train_raw['CREDIT_LIMIT'].max()

        # Padding
        x_span = np.linspace(x_min - 0.1 * abs(x_min), x_max + 0.1 * x_max, 200)
        y_span = np.linspace(y_min - 0.1 * abs(y_min), y_max + 0.1 * y_max, 200)
        xx, yy = np.meshgrid(x_span, y_span)

        # Scale grid coordinates
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        grid_points_scaled = self.scaler.transform(grid_points)

        # Predict responsibilities (probabilities)
        responsibilities = self.gmm.predict_proba(grid_points_scaled)
        max_resp = np.max(responsibilities, axis=1).reshape(xx.shape)

        return xx, yy, max_resp

    def plot_training_assignments(self):
        xx, yy, max_resp = self._generate_contour_data()

        # Predict clusters on training points
        train_clusters = self.gmm.predict(self.X_train_scaled)
        plot_df = self.X_train_raw.copy()
        plot_df['Cluster'] = train_clusters.astype(str)

        # Base contour map of soft posterior boundaries
        fig = go.Figure(data=go.Contour(
            x=xx[0, :], y=yy[:, 0], z=max_resp,
            colorscale='YlGnBu', contours_coloring='heatmap',
            colorbar=dict(title="Max Posterior Prob."),
            opacity=0.65, showscale=True
        ))

        # Overlay training points
        for cluster in sorted(plot_df['Cluster'].unique()):
            cluster_df = plot_df[plot_df['Cluster'] == cluster]
            fig.add_trace(go.Scatter(
                x=cluster_df['PURCHASES'], y=cluster_df['CREDIT_LIMIT'],
                mode='markers', name=f'Cluster {cluster}',
                marker=dict(size=5, line=dict(width=0.5, color='white'))
            ))

        fig.update_layout(
            title="GMM Training Cluster Assignments over Posterior Contours",
            xaxis_title="Purchases ($)", yaxis_title="Credit Limit ($)",
            template="plotly_white",
            legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
        )
        return fig

    def plot_test_assignments(self):
        xx, yy, max_resp = self._generate_contour_data()

        # Predict clusters on validation/test points
        val_clusters = self.gmm.predict(self.X_val_scaled)
        plot_df = self.X_val_raw.copy()
        plot_df['Cluster'] = val_clusters.astype(str)

        # Base contour map
        fig = go.Figure(data=go.Contour(
            x=xx[0, :], y=yy[:, 0], z=max_resp,
            colorscale='YlGnBu', contours_coloring='heatmap',
            colorbar=dict(title="Max Posterior Prob."),
            opacity=0.65, showscale=True
        ))

        # Overlay out-of-sample validation points
        for cluster in sorted(plot_df['Cluster'].unique()):
            cluster_df = plot_df[plot_df['Cluster'] == cluster]
            fig.add_trace(go.Scatter(
                x=cluster_df['PURCHASES'], y=cluster_df['CREDIT_LIMIT'],
                mode='markers', name=f'Cluster {cluster} (Test)',
                marker=dict(size=6, symbol='diamond', line=dict(width=0.5, color='black'))
            ))

        fig.update_layout(
            title="Out-of-Sample Test Assignments over Posterior Contours",
            xaxis_title="Purchases ($)", yaxis_title="Credit Limit ($)",
            template="plotly_white",
            legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
        )
        return fig

# Initialize and execute pipeline
segmenter = GMMFinancialSegmenter(n_components=3)
segmenter.load_and_preprocess_data()
segmenter.fit()
segmenter.evaluate()

# Generate figures
fig_density = segmenter.plot_raw_density()
fig_train = segmenter.plot_training_assignments()
fig_test = segmenter.plot_test_assignments()

# Render plots
fig_density.show()
fig_train.show()
fig_test.show()

Data successfully preprocessed.
Training set shape: (7159, 2), Validation set shape: (1790, 2)
--- EM Algorithm Execution ---
Converged: True
Number of iterations to converge: 18
Validation Set Average Log-Likelihood: -1.6890
